In [1]:
""""
Copyright (C) 2025 ETH Zurich, Stefan J.P. Chrin, Gregory Landrum, Sereina Riniker
"""

#!/usr/bin/env python
import os
import subprocess
import json
from filelock import FileLock  # For threadsafe log writes
from joblib import Parallel, delayed  


# --- Variables ---
base_dir = "../../../0_DATA/COMPOUNDS/PER_TARGET_SDF"  # Base directory containing target folders
sdf_activity = "dG"  # SDF activity parameter
comsia_script = "./pycomsia/comsia.py"  # Script to run
comsia_cmd = "conda run -n pycomsia python"  # Use conda run to ensure env is activated
n_splits = 100

output_dir = "./A_QSAR_MIF"  # Base output directory
log_path = "./"
log_file = os.path.join(log_path, "comsia_log.json")  # Log file path (JSON format)

TARGETS_TO_SKIP = ("galectin","p2","thrombin","cdk8","hif2a","hunt","eg5","jnk1")

# --- Log Functions ---
def load_log():
    try:
        with open(log_file, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return []

def save_log(log_entries):
    with open(log_file, 'w') as f:
        json.dump(log_entries, f, indent=2)

def log_exists(entry, log_entries):
    # Check uniqueness based on target, column_filter, and grid_resolution
    for e in log_entries:
        if (e["target"] == entry["target"] and 
            e["column_filter"] == entry["column_filter"] and 
            e["grid_resolution"] == entry["grid_resolution"]):
            return True
    return False

def add_log_entry(entry):
    lock = FileLock(log_file + ".lock")
    with lock:
        log_entries = load_log()
        if not log_exists(entry, log_entries):
            log_entries.append(entry)
            save_log(log_entries)

# --- Top-level function to run a single job (target + hyperparameters) ---
def run_job(target, sdf_file, cf, gr):
    entry = {
        "target": target,
        "column_filter": cf,
        "grid_resolution": gr,
        "sdf_file": sdf_file
    }
    with FileLock(log_file + ".lock"):
        current_log = load_log()
        if log_exists(entry, current_log):
            print(f"Skipping already run: {entry}")
            return
    # Create target-specific output folder
    target_output_dir = os.path.join(output_dir, target)
    os.makedirs(target_output_dir, exist_ok=True)
    # Build command 
    cmd = (
        f"{comsia_cmd} {comsia_script} "
        f"--train_file {sdf_file} "
        f"--sdf_activity {sdf_activity} "
        f"--column_filter {cf} "
        f"--grid_resolution {gr} "
        f"--output_dir {target_output_dir} "  # Use target-specific folder
        f"--n_splits {n_splits} "
        f"--disable_visualization "
    )
    # Add --fields parameter for targets pde2 or p2 because they have no H bond donors -> otherwise field will be empty causing error
    if target in ("pde2", "p2"):
        cmd += "--fields SEH "
    print(f"Running: {cmd}")
    result = subprocess.run(cmd, shell=True, text=True)
    if result.returncode != 0:
        print(f"[ERROR] Command failed: {cmd}")
        print(f"Return code: {result.returncode}")
        print(f"STDERR:\n{result.stderr}")
        print(f"STDOUT:\n{result.stdout}")
        entry["status"] = "failed"
        entry["returncode"] = result.returncode
        entry["stderr"] = result.stderr
        entry["stdout"] = result.stdout
    else:
        entry["status"] = "success"
    add_log_entry(entry)

# --- Additional parameters to iterate (Parameter search no longer used) ---
column_filters = [0.001]  # List of column filters
grid_resolutions = [1]   # List of grid resolutions

# --- Build job list over targets and hyperparameters ---
jobs = []
for target in sorted(os.listdir(base_dir)):
    if target in TARGETS_TO_SKIP:  # Skip targets
        continue
    target_path = os.path.join(base_dir, target)
    if os.path.isdir(target_path):
        sdf_file = os.path.join(target_path, f"{target}.sdf")
        if os.path.exists(sdf_file):
            for cf in column_filters:
                for gr in grid_resolutions:
                    jobs.append((target, sdf_file, cf, gr))
        else:
            print(f"SDF file not found in {target_path}")

# --- Run all jobs in parallel ---
Parallel(n_jobs=6)(
    delayed(run_job)(target, sdf_file, cf, gr)
    for target, sdf_file, cf, gr in jobs
)


Reading SDF file...

steric_field:
Original points: 4896
Points kept: 4238
Points filtered: 658

electrostatic_field:
Original points: 4896
Points kept: 4896
Points filtered: 0

hydrophobic_field:
Original points: 4896
Points kept: 1777
Points filtered: 3119

hbond_donor_field:
Original points: 4896
Points kept: 2021
Points filtered: 2875

hbond_acceptor_field:
Original points: 4896
Points kept: 1433
Points filtered: 3463
q2 = 0.3591 for component = 1
q2 = 0.5522 for component = 2
q2 = 0.5621 for component = 3
q2 = 0.6184 for component = 4
q2 = 0.5711 for component = 5
q2 = 0.5533 for component = 6
q2 = 0.5573 for component = 7
q2 = 0.5349 for component = 8
q2 = 0.4798 for component = 9
q2 = 0.4746 for component = 10
q2 = 0.4693 for component = 11
q2 = 0.4709 for component = 12
Optimal number of components based on maximum Q2: 4
CV results saved to ./A_QSAR_MIF/cdk2/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


Reading SDF file...

steric_field:
Original points: 3822
Points kept: 3822
Points filtered: 0

electrostatic_field:
Original points: 3822
Points kept: 3822
Points filtered: 0

hydrophobic_field:
Original points: 3822
Points kept: 2650
Points filtered: 1172

hbond_donor_field:
Original points: 3822
Points kept: 2886
Points filtered: 936

hbond_acceptor_field:
Original points: 3822
Points kept: 693
Points filtered: 3129
q2 = 0.1008 for component = 1
q2 = 0.3564 for component = 2
q2 = 0.4296 for component = 3
q2 = 0.3943 for component = 4
q2 = 0.4088 for component = 5
q2 = 0.4253 for component = 6
q2 = 0.3972 for component = 7
q2 = 0.3799 for component = 8
q2 = 0.3485 for component = 9
q2 = 0.3254 for component = 10
q2 = 0.3132 for component = 11
q2 = 0.3086 for component = 12
Optimal number of components based on maximum Q2: 3
CV results saved to ./A_QSAR_MIF/bace/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


Reading SDF file...

steric_field:
Original points: 4212
Points kept: 4007
Points filtered: 205

electrostatic_field:
Original points: 4212
Points kept: 4212
Points filtered: 0

hydrophobic_field:
Original points: 4212
Points kept: 2402
Points filtered: 1810
q2 = 0.4861 for component = 1
q2 = 0.4618 for component = 2
q2 = 0.4397 for component = 3
q2 = 0.3423 for component = 4
q2 = 0.1637 for component = 5
q2 = -0.0659 for component = 6
q2 = -0.1738 for component = 7
q2 = -0.4390 for component = 8
q2 = -0.6466 for component = 9
q2 = -0.7573 for component = 10
q2 = -0.7370 for component = 11
q2 = -0.6612 for component = 12
Optimal number of components based on maximum Q2: 1
CV results saved to ./A_QSAR_MIF/pde2/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


Reading SDF file...

steric_field:
Original points: 6720
Points kept: 6581
Points filtered: 139

electrostatic_field:
Original points: 6720
Points kept: 6720
Points filtered: 0

hydrophobic_field:
Original points: 6720
Points kept: 3859
Points filtered: 2861

hbond_donor_field:
Original points: 6720
Points kept: 3987
Points filtered: 2733

hbond_acceptor_field:
Original points: 6720
Points kept: 1705
Points filtered: 5015
q2 = 0.4357 for component = 1
q2 = 0.4468 for component = 2
q2 = 0.4175 for component = 3
q2 = 0.3509 for component = 4
q2 = 0.3717 for component = 5
q2 = 0.4373 for component = 6
q2 = 0.4851 for component = 7
q2 = 0.4894 for component = 8
q2 = 0.4990 for component = 9
q2 = 0.5180 for component = 10
q2 = 0.5165 for component = 11
q2 = 0.5012 for component = 12
Optimal number of components based on maximum Q2: 10
CV results saved to ./A_QSAR_MIF/cmet/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


Reading SDF file...

steric_field:
Original points: 8448
Points kept: 8155
Points filtered: 293

electrostatic_field:
Original points: 8448
Points kept: 8448
Points filtered: 0

hydrophobic_field:
Original points: 8448
Points kept: 3822
Points filtered: 4626

hbond_donor_field:
Original points: 8448
Points kept: 4067
Points filtered: 4381

hbond_acceptor_field:
Original points: 8448
Points kept: 1669
Points filtered: 6779
q2 = 0.0576 for component = 1
q2 = 0.1828 for component = 2
q2 = -0.1228 for component = 3
q2 = -0.1973 for component = 4
q2 = 0.1051 for component = 5
q2 = -0.0808 for component = 6
q2 = -0.1534 for component = 7
q2 = -0.1641 for component = 8
q2 = -0.1703 for component = 9
q2 = -0.2001 for component = 10
q2 = -0.1063 for component = 11
q2 = -0.0543 for component = 12
Optimal number of components based on maximum Q2: 2
CV results saved to ./A_QSAR_MIF/pde10/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


...

Reading SDF file...

steric_field:
Original points: 4508
Points kept: 4504
Points filtered: 4

electrostatic_field:
Original points: 4508
Points kept: 4508
Points filtered: 0

hydrophobic_field:
Original points: 4508
Points kept: 3287
Points filtered: 1221

hbond_donor_field:
Original points: 4508
Points kept: 3264
Points filtered: 1244

hbond_acceptor_field:
Original points: 4508
Points kept: 2041
Points filtered: 2467
q2 = 0.0702 for component = 1
q2 = 0.1485 for component = 2
q2 = 0.1519 for component = 3
q2 = 0.2198 for component = 4
q2 = 0.1958 for component = 5
q2 = -0.0333 for component = 6
q2 = -0.0798 for component = 7
q2 = -0.0828 for component = 8
q2 = 0.0225 for component = 9
q2 = 0.0196 for component = 10
q2 = 0.0891 for component = 11
q2 = 0.0846 for component = 12
Optimal number of components based on maximum Q2: 4
CV results saved to ./A_QSAR_MIF/tnks2/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


Reading SDF file...

steric_field:
Original points: 3920
Points kept: 3098
Points filtered: 822

electrostatic_field:
Original points: 3920
Points kept: 3920
Points filtered: 0

hydrophobic_field:
Original points: 3920
Points kept: 1260
Points filtered: 2660

hbond_donor_field:
Original points: 3920
Points kept: 1472
Points filtered: 2448

hbond_acceptor_field:
Original points: 3920
Points kept: 850
Points filtered: 3070
q2 = 0.0399 for component = 1
q2 = 0.2339 for component = 2
q2 = 0.3087 for component = 3
q2 = 0.3179 for component = 4
q2 = 0.4106 for component = 5
q2 = 0.5170 for component = 6
q2 = 0.5869 for component = 7
q2 = 0.6435 for component = 8
q2 = 0.6864 for component = 9
q2 = 0.6539 for component = 10
q2 = 0.5657 for component = 11
q2 = 0.5586 for component = 12
Optimal number of components based on maximum Q2: 9
CV results saved to ./A_QSAR_MIF/tyk2/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


Reading SDF file...

steric_field:
Original points: 7182
Points kept: 6864
Points filtered: 318

electrostatic_field:
Original points: 7182
Points kept: 7182
Points filtered: 0

hydrophobic_field:
Original points: 7182
Points kept: 3839
Points filtered: 3343

hbond_donor_field:
Original points: 7182
Points kept: 4049
Points filtered: 3133

hbond_acceptor_field:
Original points: 7182
Points kept: 2285
Points filtered: 4897
q2 = -0.0219 for component = 1
q2 = -0.0069 for component = 2
q2 = 0.0055 for component = 3
q2 = -0.0315 for component = 4
q2 = -0.0286 for component = 5
q2 = -0.0709 for component = 6
q2 = -0.0339 for component = 7
q2 = 0.0041 for component = 8
q2 = 0.0437 for component = 9
q2 = 0.0624 for component = 10
q2 = 0.1091 for component = 11
q2 = 0.1214 for component = 12
Optimal number of components based on maximum Q2: 12
CV results saved to ./A_QSAR_MIF/pfkfb3/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


Reading SDF file...

steric_field:
Original points: 7728
Points kept: 7482
Points filtered: 246

electrostatic_field:
Original points: 7728
Points kept: 7728
Points filtered: 0

hydrophobic_field:
Original points: 7728
Points kept: 4822
Points filtered: 2906

hbond_donor_field:
Original points: 7728
Points kept: 4978
Points filtered: 2750

hbond_acceptor_field:
Original points: 7728
Points kept: 3958
Points filtered: 3770
q2 = 0.2498 for component = 1
q2 = 0.2839 for component = 2
q2 = 0.3552 for component = 3
q2 = 0.4012 for component = 4
q2 = 0.4285 for component = 5
q2 = 0.4806 for component = 6
q2 = 0.4964 for component = 7
q2 = 0.4576 for component = 8
q2 = 0.4208 for component = 9
q2 = 0.4223 for component = 10
q2 = 0.3929 for component = 11
q2 = 0.3474 for component = 12
Optimal number of components based on maximum Q2: 7
CV results saved to ./A_QSAR_MIF/syk/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]